# Project 3 — Notebook 12: Business Summary & Recommendations
### Zone-Level Operational Benchmark — Priority-Adjusted Zone Rankings

---

| | |
|---|---|
| **Scope** | NCR Priority & SLA matrix · Mix-adjusted MTTR · Cross-zone benchmark scorecard |
| **Feeds from** | NB10 (Priority & SLA Matrix) · NB11 (Benchmark Scorecard) |
| **Audience** | Operations Leadership / Portfolio Review |

---

### What Project 3 Investigated

| Question | Notebook | Status |
|----------|----------|--------|
| Does priority mix differ by zone enough to distort raw SLA comparisons? | NB10 | ✅ |
| Which P-U tiers are driving SLA breaches in each zone? | NB10 | ✅ |
| Is MTTR variance partly explained by handling a harder ticket mix? | NB10 | ✅ |
| Cross-zone benchmark scorecard on a level playing field | NB11 | ✅ |

## 1. Setup

In [1]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline

os.chdir(os.path.join('..', '..'))
if os.path.abspath(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from src.fault_ticket.metrics import calculate_zone_summary
from config import ZONE_ORDER, ZONE_PALETTE, SLA_THRESHOLDS

df      = pd.read_csv('output/cleaned_fault_ticket.csv')
df_zone = df[(df['ZONE'].isin(ZONE_ORDER)) & (df['Priority'] < 4)].copy()
summary = calculate_zone_summary(df[df['Priority'] < 4]).set_index('ZONE').reindex(ZONE_ORDER)
# Priority ≥ 4 excluded to match df_zone filter

# Priority ≥ 4 excluded — P4+ tickets are preventive maintenance (scheduled work),
# not reactive faults. Including them distorts MTTR, SLA rates, and priority mix.
# P1: national core · P2: regional core · P3: zone equipment (BTS-level)
# Urgency: 1=Outage · 2=Degradation · 3=Management Alarm
P_LABEL = {1:'P1 — National Core (3h)', 2:'P2 — Regional Core (6h)', 3:'P3 — Zone Equipment (9-24h)'}
p_order = ['P1 — National Core (3h)', 'P2 — Regional Core (6h)', 'P3 — Zone Equipment (9-24h)']
df_zone['P_Label'] = df_zone['Priority'].map(P_LABEL)
df_pu   = df_zone.copy()

# ── Re-derive scorecard dimensions ────────────────────────────────────────
priority_mix = (df_zone.groupby(['ZONE','P_Label']).size()
                 .unstack(fill_value=0)
                 .reindex(columns=p_order, fill_value=0)
                 .reindex(ZONE_ORDER))
priority_mix_pct = priority_mix.div(priority_mix.sum(axis=1), axis=0) * 100
ncr_p_mix = priority_mix_pct.mean()

clean_fd = df_zone[(df_zone['Resolution_Path']=='Field_Dispatch_Restored') &
                   (df_zone['Timestamp_Integrity'])].copy()

dim_sla       = df_pu.groupby('ZONE')['SLA_Compliant'].mean().mul(100).reindex(ZONE_ORDER)
dim_high_sla  = (df_pu[df_pu['Priority'].isin([1,2])]
                 .groupby('ZONE')['SLA_Compliant'].mean().mul(100).reindex(ZONE_ORDER))
dim_field_adj = (clean_fd.groupby(['ZONE','P_Label'])['FIELD_TIME_HOURS'].mean()
                  .unstack(fill_value=np.nan).reindex(columns=p_order, fill_value=np.nan)
                  .reindex(ZONE_ORDER).mul(ncr_p_mix/100).sum(axis=1))
dim_density   = summary['Fault_Density']
dim_mttr_adj  = (df_zone.groupby(['ZONE','P_Label'])['OUTAGEDURATION'].mean()
                  .unstack(fill_value=np.nan).reindex(columns=p_order, fill_value=np.nan)
                  .reindex(ZONE_ORDER).mul(ncr_p_mix/100).sum(axis=1))

def norm(s, invert=False):
    mn, mx = s.min(), s.max()
    n = (s - mn) / (mx - mn + 1e-9) * 100
    return (100 - n) if invert else n

WEIGHTS = {'SLA_Score':0.30,'HighPrio_Score':0.25,
           'Field_Score':0.20,'Density_Score':0.15,'MTTR_Score':0.10}
scorecard = pd.DataFrame({
    'SLA_Score'      : norm(dim_sla),
    'HighPrio_Score' : norm(dim_high_sla),
    'Field_Score'    : norm(dim_field_adj, invert=True),
    'Density_Score'  : norm(dim_density,   invert=True),
    'MTTR_Score'     : norm(dim_mttr_adj,  invert=True),
}, index=ZONE_ORDER).round(1)
scorecard['Composite'] = sum(scorecard[c]*w for c,w in WEIGHTS.items()).round(1)
scorecard['Rank'] = scorecard['Composite'].rank(ascending=False).astype(int)

z = {row['ZONE']: row for _, row in calculate_zone_summary(df).iterrows()}
z1,z2,z3,z4,z5,z6 = (z.get(f'ZONE {i}') for i in [1,2,3,4,5,6])


top_zone  = scorecard['Composite'].idxmax()
bot_zone  = scorecard['Composite'].idxmin()
p1p2_max  = (priority_mix_pct['P1 — National Core (3h)']+priority_mix_pct['P2 — Regional Core (6h)']).idxmax()
p1p2_min  = (priority_mix_pct['P1 — National Core (3h)']+priority_mix_pct['P2 — Regional Core (6h)']).idxmin()

print(f"✅ Scorecard ready  |  #1: {top_zone} ({scorecard.loc[top_zone,'Composite']:.1f})  "
      f"|  #6: {bot_zone} ({scorecard.loc[bot_zone,'Composite']:.1f})")
print(f"   Highest P1+P2 share: {p1p2_max}  |  Lowest: {p1p2_min}")

✅ Scorecard ready  |  #1: ZONE 4 (87.4)  |  #6: ZONE 5 (19.9)
   Highest P1+P2 share: ZONE 4  |  Lowest: ZONE 3


## 2. Key Metrics Snapshot

In [2]:
metrics = pd.DataFrame({
    'Metric': [
        'Benchmark leader (composite score)',
        'Most improved vs. raw ranking',
        'Zone with highest National+Regional Core (P1+P2) share',
        'Zone with lowest National+Regional Core (P1+P2) share',
        'Largest SLA breach concentration (tier)',
        'Mix-adjusted MTTR range (best → worst)',
    ],
    'Value': [
        f"{top_zone}  (score: {scorecard.loc[top_zone,'Composite']:.1f})",
        f"See NB11 — raw vs adjusted ranking comparison",
        f"{p1p2_max}  ({(priority_mix_pct.loc[p1p2_max,'P1 — National Core (3h)']+priority_mix_pct.loc[p1p2_max,'P2 — Regional Core (6h)']):.1f}% of zone tickets)",
        f"{p1p2_min}  ({(priority_mix_pct.loc[p1p2_min,'P1 — National Core (3h)']+priority_mix_pct.loc[p1p2_min,'P2 — Regional Core (6h)']):.1f}% of zone tickets)",
        f"P3 (Zone Equipment) — most breach volume; see heatmap for P3.1 (9h) vs P3.2 (12h) vs P3.3 (24h) split",
        f"{dim_mttr_adj.min():.0f}h ({dim_mttr_adj.idxmin()}) → {dim_mttr_adj.max():.0f}h ({dim_mttr_adj.idxmax()})",
    ]
})
display(metrics.style.hide(axis='index')
    .set_caption('Table 1 — Project 3 Key Metrics Snapshot')
    .set_properties(**{'text-align':'left'})
    .set_table_styles([
        {'selector':'caption','props':[('font-size','13px'),('font-weight','bold'),
                                        ('text-align','left'),('padding-bottom','6px')]},
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('font-size','12px'),('padding','8px 12px')]},
        {'selector':'td','props':[('padding','7px 12px'),('font-size','12px')]},
        {'selector':'tr:nth-child(even)','props':[('background-color','#f7f9fc')]},
    ])
)

Metric,Value
Benchmark leader (composite score),ZONE 4 (score: 87.4)
Most improved vs. raw ranking,See NB11 — raw vs adjusted ranking comparison
Zone with highest National+Regional Core (P1+P2) share,ZONE 4 (16.1% of zone tickets)
Zone with lowest National+Regional Core (P1+P2) share,ZONE 3 (2.5% of zone tickets)
Largest SLA breach concentration (tier),P3 (Zone Equipment) — most breach volume; see heatmap for P3.1 (9h) vs P3.2 (12h) vs P3.3 (24h) split
Mix-adjusted MTTR range (best → worst),36h (ZONE 3) → 81h (ZONE 5)


## 3. Key Findings

### Finding 1 🟠 — Priority Mix Differs Meaningfully by Zone

The P1+P2 share (National Core + Regional Core tickets with ≤6h SLA targets) varies
significantly across zones. Zones with a higher share of core-network tickets face a
structurally harder operating environment — these are outages and degradations on BSC
and core transmission equipment, not BTS-level zone faults.
Raw SLA compliance rates — which don't account for this — overstate the
performance gap between zones with easy and hard ticket mixes.

The priority-adjusted benchmark in NB11 controls for this: zones that maintain strong
SLA rates on a harder P1+P2 ticket mix receive higher scores than zones achieving
similar rates on softer mixes. This is the correct basis for cross-zone comparison.

Zone 4 and Zone 5 carry the highest core ticket share (16.1% and 15.6% P1+P2 respectively), while Zone 3 is almost entirely P3 equipment tickets (97.5%). Notably, Zone 3 has the highest field dispatch rate (86.1%) yet the lowest MTTR (32.3h) — demonstrating that high field dispatch does not imply slow resolution when the ticket mix is predominantly routine BTS-level faults with predictable repair patterns.

---

### Finding 2 🔴 — P3 Tier Drives the Most Breach Volume Across All Zones

P3 tickets (Zone Equipment — BTS serving a specific area) account for the vast majority of volume (83–97% per zone) and therefore drive most breach counts. P3 urgency matters critically here: P3.2 (Degradation · 12h) consistently shows the highest breach rates across all zones — ranging from 31.6% (Zone 2) to 44.2% (Zone 6). P3.1 (Outage · 9h) breach rates are lower (10.1–19.9%) suggesting the 9h outage target, while tight, is more consistently met than the 12h degradation target.

P3.2's high breach rate likely reflects that degradation alarms are deprioritised relative to outages — engineers resolve outages first, leaving degradation tickets to age past 12h. Under the 'Site ko, sagot ko' accountability model, if the assigned engineer is on day-off when a degradation alarm fires, the endorsement-to-dispatch delay alone can consume a significant portion of the 12h window before a field technician even arrives on site.

---

### Finding 3 🟢 — Mix-Adjusted Ranking Shifts Zone 4's Benchmark Status

On a raw SLA basis Zone 4 leads. On a priority-adjusted, field-efficient basis the
scorecard confirms Zone 4 as the operational benchmark — it handles a high P1+P2 share
in a high-CBD environment and still leads on SLA and P1+P2 compliance. Its benchmark
status is not an artifact of an easy ticket mix.


## 4. Recommendations

### Immediate (0–3 Months)

**REC-P3-02 · Address P3.2 Degradation Alarm Breach Rate**
P3.2 tickets (BTS-level degradation alarms · 12h target) have the highest breach rates across all zones (31.6–44.2%). P3.1 outages (9h target) are better-controlled (10.1–19.9% breach). The likely mechanism: engineers prioritise outages over degradations, and under the 'Site ko, sagot ko' model, day-off endorsements for degradation alarms can consume hours of the 12h window before dispatch even begins.

Recommended actions: (a) treat P3.2 endorsement as a same-urgency handoff with a maximum 2h endorsement-to-dispatch SLA for the duty team lead; (b) track P3.2 endorsement delay separately in reporting to isolate accountability-model friction from actual repair time; (c) consider whether the 12h P3.2 target should be extended to 18h to reflect the two-stage dispatch reality, or whether faster endorsement protocols are the correct fix.

---

### Short-Term (3–6 Months)

**REC-P3-03 · Use Priority-Adjusted Scorecard for Zone Performance Reviews**
Replace raw SLA compliance rate as the single KPI in zone performance reviews with
the composite scorecard from NB11. This ensures zones handling harder ticket mixes
are not penalised and zones with easy mixes are not over-credited.

**REC-P3-04 · Zone 5 — Address Highest MTTR and P3.2 Breach Rate**
Zone 5 has the worst MTTR in NCR (77.8h raw · 81.3h mix-adjusted) and carries the highest field time (80.7h avg on timestamp-clean tickets). Its 15.6% P1+P2 core share is the second-highest, but the dominant driver of performance lag appears to be P3.2 degradation breach (38.9%) and CBD-related field delay in Makati and BGC sites.

Recommended actions: (a) under 'Site ko, sagot ko', Zone 5 engineers assigned to CBD sites should have mandatory pre-arranged building access on file — day-off endorsements for these sites must include access coordination handoff, not just work order transfer; (b) the Zone 5 area head should review P3.2 breach tickets monthly to identify whether breach root cause is endorsement delay, travel time, or permit friction; (c) consider a dedicated tandem arrangement for high-volume CBD sites where a second engineer is pre-briefed on the site layout.

**REC-P3-05 · Resolve UNKNOWN-Under Investigation Tickets**
13.9% of NCR tickets (5,129) carry 'UNKNOWN-Under Investigation' as their RFO. These are excluded from RFO-level analysis as their root cause is unresolved. Their anomalously high SLA compliance (92.7%) suggests many may be auto-closed or timed-out without proper investigation. The area head and team leads should institute a monthly closure review: any ticket open beyond 30 days must have a confirmed RFO before closure. This will improve RFO data quality for all future analysis.

---

### Strategic (6–12 Months)

**REC-P3-06 · Proceed to Project 4 — City-Level Intelligence**
The benchmark scorecard establishes zone-level rankings. Project 4 will decompose
performance to city level within each zone, identifying which specific cities are
driving zone-level underperformance and where targeted interventions will have the
highest leverage.

## 5. Next Steps

| Priority | Action | Timeline |
|----------|--------|----------|
| 🔴 High | Address P3.2 degradation breach (31–44% across zones) — endorsement-to-dispatch SLA | Month 1 |
| 🔴 High | Zone 5 — CBD site access handoff protocol for day-off endorsements | Month 1–2 |
| 🟡 Medium | Adopt priority-adjusted scorecard for zone performance reviews | Month 2 |
| 🟡 Medium | Resolve UNKNOWN-Under Investigation tickets — mandatory RFO closure review | Month 2–3 |
| 🟢 Standard | Zone 4 — document benchmark practices for cross-zone knowledge transfer | Month 3–4 |
| ▶ Next | **Commence Project 4: City-Level Intelligence** | Month 2 |

---

> **Project 4 will answer:** Which cities within each zone are driving zone-level
> underperformance? Is Zone 5's elevated MTTR (77.8h) and P3.2 breach rate concentrated
> in Makati and BGC CBD sites, or spread zone-wide? Which cities have the highest
> UNKNOWN-Under Investigation ticket rates? Where is the 'Site ko, sagot ko' endorsement
> delay most severe — and which team leads are the bottleneck? What city-level
> interventions would most efficiently move the zone composite score?